In [23]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd

html_path = Path("/Users/HP/Documents/scrapping/eurojob.html")
raw_html = html_path.read_text(encoding="utf-8", errors="ignore").strip()

# File is saved as [<html...>] from notebook output; strip wrappers.
if raw_html.startswith("[") and raw_html.endswith("]"):
    raw_html = raw_html[1:-1]

soup = BeautifulSoup(raw_html, "html.parser")

#save the soup to a html file
with open("eurojob.html", "w", encoding="utf-8") as f:
    f.write(soup.prettify())

In [ ]:
# imports
import pandas as pd

# read html file
job_items = soup.select("ul.searchList > li")

jobs = []
for item in job_items:
    title_el = item.select_one("h3 a")
    company_el = item.select_one(".companyName")
    location_el = item.select_one(".location")
    posted_el = item.select_one(".postedDate")
    summary_el = item.select_one("p")

    # Keep only true jobs; skip alert/signup/noise blocks.
    if not (title_el and company_el and location_el):
        continue

    jobs.append({
        "title": title_el.get_text(strip=True),
        "company": company_el.get_text(strip=True),
        "location": location_el.get_text(strip=True),
        "posted": posted_el.get_text(" ", strip=True) if posted_el else "",
        "summary": summary_el.get_text(" ", strip=True) if summary_el else "",
        "url": title_el.get("href", "")
    })

df_jobs = pd.DataFrame(jobs)
df_jobs = df_jobs.drop_duplicates(subset=["title", "company", "location"]).reset_index(drop=True)

print(f"Total blocks in searchList: {len(job_items)}")
print(f"Filtered job rows: {len(df_jobs)}")
df_jobs.head(20)

In [21]:
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     112 non-null    object
 1   company   112 non-null    object
 2   location  112 non-null    object
 3   posted    112 non-null    object
 4   summary   112 non-null    object
 5   url       112 non-null    object
dtypes: object(6)
memory usage: 5.4+ KB


In [ ]:
output_csv = Path("/Users/HP/Documents/scrapping/jobs_filtered.csv")
df_jobs.to_csv(output_csv, index=False)
output_csv


In [ ]:
# inspect the euroengineerjobs_redo.csv
# scapping has been redone using "euroengineer_scrapper.py" and saved as "euroengineerjobs_redo.csv"
df_euroengineer_csv = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/euroengineerjobs_redo.csv")

# count rows with empy summary
empty_summary_count = df_euroengineer_csv["summary"].isna().sum()
print(f"Number of rows with empty summary: {empty_summary_count}")

#drop rows with empty summary
df_euroengineer_csv = df_euroengineer_csv.dropna(subset=["summary"]).reset_index(drop=True)

#save new csv
output_csv_redo = Path("/Users/HP/Documents/scrapping/scrapped_jobs/euroengineerjobs_redo_no_empty_summary.csv")
df_euroengineer_csv.to_csv(output_csv_redo, index=False)
output_csv_redo


Number of rows with empty summary: 89


PosixPath('/Users/HP/Documents/scrapping/scrapped_jobs/euroengineerjobs_redo_no_empty_summary.csv')

In [161]:
# inspect the euroengineerjobs_redo.csv
import pandas as pd
df_euroengineer = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/euroengineerjobs_redo.csv")
#df_euroengineer.head()
# count rows with empy summary
empty_summary_count = df_euroengineer["summary"].isna().sum()
print(f"Number of rows with empty summary: {empty_summary_count}")

Number of rows with empty summary: 217


In [ ]:

#read csv
df_jobs_energyjob = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/energyjobsearch_jobs.csv")
df_jobs_energyjob.head(20)




In [48]:
# euro climate jobs

# the redo on eurojobsearch
import pandas as pd
from bs4 import BeautifulSoup
import requests
url = "https://www.euroclimatejobs.com/job_search/category/bioenergy_and_biofuel/category/carbon_capture_and_storage/category/climate_and_energy_analyst/category/consultant/category/electricity/category/energy_engineer/category/geothermal_energy/category/government_and_associations/category/hydropower_energy/category/oil_and_gas/category/policy_and_regulation/category/production_and_operations/category/project_manager/category/renewable_energy/category/research_and_development/category/solar_energy/category/sustainable_transport/category/wave_and_tidal_energy/category/wind_energy/experience/3-4_years_experience/experience/5+_years_experience/experience/manager_and_executive"

page = requests.get(url) 

soup_euroclimatejobs = BeautifulSoup(page.content, "html")

#save the soup to a html file
with open("euroclimatejobs.html", "w", encoding="utf-8") as f:
    f.write(soup_euroclimatejobs.prettify())   


In [ ]:
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

html_path = Path("/Users/HP/Documents/scrapping/euroclimatejobs.html")
base_url = "https://www.euroclimatejobs.com"

html = html_path.read_text(encoding="utf-8", errors="ignore")
soup = BeautifulSoup(html, "html.parser")

rows = []
for li in soup.select("ul.searchList > li"):
    title_el = li.select_one("h3 a")
    company_el = li.select_one(".companyName")
    location_el = li.select_one(".location")

    # keep only real job cards; skip alert/subscribe blocks
    if not (title_el and company_el and location_el):
        continue

    posted_el = li.select_one(".postedDate")
    save_el = li.select_one("a.jobSave[data-job-id]")

    # summary: first paragraph with meaningful text
    summary = ""
    for p in li.select("p"):
        txt = p.get_text(" ", strip=True)
        if txt and "Email me jobs like this" not in txt and "Subscribe" not in txt:
            summary = txt
            break

    tags = [t.get_text(" ", strip=True) for t in li.select("span.badge.bg-job-tag")]

    href = title_el.get("href", "").strip()
    rows.append({
        "job_id": save_el.get("data-job-id") if save_el else "",
        "title": title_el.get_text(" ", strip=True),
        "company": company_el.get_text(" ", strip=True),
        "location": location_el.get_text(" ", strip=True),
        "posted": posted_el.get_text(" ", strip=True) if posted_el else "",
        "summary": summary,
        "job_url": urljoin(base_url, href),
        "tags": " | ".join(tags),
    })

df = pd.DataFrame(rows).drop_duplicates(subset=["job_id", "title", "company", "location"]).reset_index(drop=True)

print(f"Extracted jobs: {len(df)}")
print(df.head(10))

out_csv = Path("/Users/HP/Documents/scrapping/scrapped_jobs/euroclimatejobs_jobs.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
df.info()


In [177]:
# inspect the euroclimatejobs.csv
import pandas as pd
df_euroclimate = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/euroclimatejobs_jobs.csv")
#df_euroclimate.head()
# count rows with empy summary
empty_summary_count = df_euroclimate["summary"].isna().sum()
print(f"Number of rows with empty summary: {empty_summary_count}")

#save new csv without empty summary
df_euroclimate_no_empty_summary = df_euroclimate.dropna(subset=["summary"]).reset_index(drop=True)
output_csv_no_empty_summary = Path("/Users/HP/Documents/scrapping/scrapped_jobs/euroclimatejobs_jobs_no_empty_summary.csv")
df_euroclimate_no_empty_summary.to_csv(output_csv_no_empty_summary, index=False)
print(f"Saved: {output_csv_no_empty_summary}")

Number of rows with empty summary: 6
Saved: /Users/HP/Documents/scrapping/scrapped_jobs/euroclimatejobs_jobs_no_empty_summary.csv


In [ ]:
# check dataframe of rejobs
import pandas as pd
df_rejobs = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/rejobs_jobs.csv")
df_rejobs.info()
df_rejobs.head(20)

In [ ]:
# identify rows which their tags does not contain word "energy"
import os

# create a filtered_folder if it doesn't exist
filtered_folder = Path("/Users/HP/Documents/scrapping/filtered_jobs_evp")
filtered_folder.mkdir(exist_ok=True)
# This creates a boolean Series (True/False for each row)
contains_energy = df_rejobs["tags"].str.contains("energy", case=False, na=False)


# rows which contains word "energy" or "voltage"
contains_energy_voltage = df_rejobs["tags"].str.contains("energy", case=False, na=False) | df_rejobs["tags"].str.contains("voltage", case=False, na=False)

# rows which contains word "energy" or "voltage" or "power"
contains_energy_voltage_power = df_rejobs["tags"].str.contains("energy", case=False, na=False) | df_rejobs["tags"].str.contains("voltage", case=False, na=False) | df_rejobs["tags"].str.contains("power", case=False, na=False)

# save to new csv df "energy, voltage, power"
filtered_df = df_rejobs[contains_energy_voltage_power]
output_path = filtered_folder / "rejobs_evp.csv"
filtered_df.to_csv(output_path, index=False)

print(f"Saved {len(filtered_df)} rows to {output_path}")

In [ ]:
# working on energyjobsearch_jobs.csv
df_energyjob = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/energyjobsearch_jobs.csv")

# filter using the short_description column for words "energy", "voltage", or "power"
contains_evp_energyjob = df_energyjob["short_description"].str.contains("energy", case=False, na=False) | df_energyjob["short_description"].str.contains("voltage", case=False, na=False) | df_energyjob["short_description"].str.contains("power", case=False, na=False)

print(f"Jobs with 'energy', 'voltage', or 'power': {contains_evp_energyjob.sum()}")
df_ejs_evp = df_energyjob[contains_evp_energyjob]
df_ejs_evp.head(20)
df_ejs_evp.to_csv("/Users/HP/Documents/scrapping/filtered_jobs_evp/energyjobsearch_evp.csv", index=False)

In [ ]:
# working on euroclimatejobs_jobs.csv
df_euroclimatejobs = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/euroclimatejobs_jobs_no_empty_summary.csv")

# filter using the tags column for words "energy", "voltage", or "power"
contains_evpeuroclimate = df_euroclimatejobs["tags"].str.contains("energy", case=False, na=False) | df_euroclimatejobs["tags"].str.contains("voltage", case=False, na=False) | df_euroclimatejobs["tags"].str.contains("power", case=False, na=False)
df_evpeuroclimate = df_euroclimatejobs[contains_evpeuroclimate]
df_evpeuroclimate.info()

#print(f"Jobs with 'energy', 'voltage', or 'power': {contains_evpeuroclimate.sum()}")
df_evpeuroclimate.to_csv("/Users/HP/Documents/scrapping/filtered_jobs_evp/euroclimatejobs_evp.csv", index=False)


In [ ]:
# working on euroengieerjobs.csv
df_euroengineerjobs = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/euroengineerjobs_redo_no_empty_summary.csv")
df_euroengineerjobs.head(20)

# filter using the summary column for words "energy", "voltage", or "power"
contains_evpeuroengineer = (
    (df_euroengineerjobs["summary"].str.contains("energy|voltage|power", case=False, na=False)) |
    (df_euroengineerjobs["title"].str.contains("energy|voltage|power|engineer", case=False, na=False))
)
df_evpeuroengineer = df_euroengineerjobs[contains_evpeuroengineer]
df_evpeuroengineer.info()
df_evpeuroengineer.to_csv("/Users/HP/Documents/scrapping/filtered_jobs_evp/euroengineer_evp.csv", index=False)

In [ ]:
df_rejobs = pd.read_csv("/Users/HP/Documents/scrapping/scrapped_jobs/rejobs_jobs.csv")
df_rejobs.info()

In [ ]:
# canonical conversion for rejobs_jobs.csv from Rejobs website
import pandas as pd
from pathlib import Path
import os

#create standardized folder if it doesn't exist
standardized_folder = Path("/Users/HP/Documents/scrapping/standardized_jobs")
standardized_folder.mkdir(exist_ok=True)

# --- 1) Load one source CSV ---
source_path = Path("/Users/HP/Documents/scrapping/filtered_jobs_evp/rejobs_evp.csv")
df = pd.read_csv(source_path)

# --- 2) Rename/map source columns to canonical names ---
# Example mapping for rejobs_jobs.csv
rename_map = {
    "employer": "company"
}
df = df.rename(columns=rename_map)

# Optional metadata columns
df["source"] = "rejobs"
df["raw_source_file"] = source_path.name

# canonical schema 
canonical_cols = [
    "job_url",
    "source",
    "title",
    "company",
    "location",
    "description",
    "seniority",
    "tags",
    "raw_source_file"
]

# --- 3) Add missing canonical columns as empty values ---
for col in canonical_cols:
    if col not in df.columns:
        df[col] = pd.NA

# --- 4) Reorder columns to canonical order and save as *_standardized.csv ---
df_standardized = df[canonical_cols]
out_path = standardized_folder / f"{source_path.stem}_std.csv"
df_standardized.to_csv(out_path, index=False)

print(f"Saved standardized file: {out_path}")
print(df_standardized.head(3))


In [ ]:
# data structure of energyjobsearch_evp.csv
df_energyjob = pd.read_csv("/Users/HP/Documents/scrapping/filtered_jobs_evp/energyjobsearch_evp.csv")
df_energyjob.info()

In [ ]:
# canonical conversion for energyjobsearch_evp.csv from EnergyJobSearch website #output saved to standardized folder as energyjobsearch_evp_std.csv
import pandas as pd
from pathlib import Path

# --- 1) Load one source CSV ---
source_path = Path("/Users/HP/Documents/scrapping/filtered_jobs_evp/energyjobsearch_evp.csv")
df = pd.read_csv(source_path)

# --- 2) Rename/map source columns to canonical names ---
# Example mapping for energyjobsearch_evp.csv
rename_map = {
    "short_description": "description",
    "url": "job_url",
}
df = df.rename(columns=rename_map)

# Build company from both company_name and partner_name.
# If both exist, keep company_name only; if one exists, use that one.
def _clean(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if not value:
        return ""
    if value.lower() in {"-", "--", "n/a", "na", "none", "null"}:
        return ""
    return value

def _merge_company(row):
    company_name = _clean(row.get("company_name"))
    partner_name = _clean(row.get("partner_name"))

    if company_name:
        return company_name
    if partner_name:
        return partner_name
    return ""

df["company"] = df.apply(_merge_company, axis=1)
df.loc[df["company"] == "", "company"] = pd.NA

# Optional metadata columns
df["source"] = "energyjobsearch"
df["raw_source_file"] = source_path.name

# Build a canonical location if not already present.
if "location" not in df.columns:
    location_parts = [c for c in ["city", "region", "country"] if c in df.columns]
    if location_parts:
        df["location"] = df[location_parts].apply(
            lambda r: ", ".join([str(v).strip() for v in r if pd.notna(v) and str(v).strip()]),
            axis=1,
        )
        df.loc[df["location"] == "", "location"] = pd.NA
    else:
        df["location"] = pd.NA

# Build tags from available fields if not already present.
if "tags" not in df.columns:
    tag_parts = [c for c in ["category", "occupation_type"] if c in df.columns]
    if tag_parts:
        df["tags"] = df[tag_parts].apply(
            lambda r: " | ".join([str(v).strip() for v in r if pd.notna(v) and str(v).strip()]),
            axis=1,
        )
        df.loc[df["tags"] == "", "tags"] = pd.NA
    else:
        df["tags"] = pd.NA

# canonical schema
canonical_cols = [
    "job_url",
    "source",
    "title",
    "company",
    "location",
    "description",
    "seniority",
    "tags",
    "raw_source_file",
]

# --- 3) Add missing canonical columns as empty values ---
for col in canonical_cols:
    if col not in df.columns:
        df[col] = pd.NA

# --- 4) Reorder columns to canonical order and save as *_standardized.csv ---
df_standardized = df[canonical_cols]
out_path = standardized_folder / f"{source_path.stem}_std.csv"
df_standardized.to_csv(out_path, index=False)

print(f"Saved standardized file: {out_path}")
print(df_standardized.head(3))
df_standardized.info()

In [180]:
df_euroclimatejobs = pd.read_csv("/Users/HP/Documents/scrapping/filtered_jobs_evp/euroclimatejobs_evp.csv")
df_euroclimatejobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   job_id    75 non-null     int64 
 1   title     75 non-null     object
 2   company   75 non-null     object
 3   location  75 non-null     object
 4   posted    75 non-null     object
 5   summary   75 non-null     object
 6   job_url   75 non-null     object
 7   tags      75 non-null     object
 8   source    75 non-null     object
dtypes: int64(1), object(8)
memory usage: 5.4+ KB


In [ ]:
# canonical conversion for euroclimate_evp.csv from euroclimate jobs website
import pandas as pd
from pathlib import Path

out_path = standardized_folder
# --- 1) Load one source CSV ---
source_path = Path("/Users/HP/Documents/scrapping/filtered_jobs_evp/euroclimatejobs_evp.csv")
df = pd.read_csv(source_path)

# --- 2) Rename/map source columns to canonical names ---
# Example mapping for euroclimatejobs_evp.csv
rename_map = {
    "summary": "description"
}
df = df.rename(columns=rename_map)

# Optional metadata columns
df["source"] = "euroclimatejobs"
df["raw_source_file"] = source_path.name

# canonical schema 
canonical_cols = [
    "job_url",
    "source",
    "title",
    "company",
    "location",
    "description",
    "seniority",
    "tags",
    "raw_source_file"
]

# --- 3) Add missing canonical columns as empty values ---
for col in canonical_cols:
    if col not in df.columns:
        df[col] = pd.NA

# --- 4) Reorder columns to canonical order and save as *_standardized.csv ---
df_standardized = df[canonical_cols]
out_path = standardized_folder / f"{source_path.stem}_std.csv"
df_standardized.to_csv(out_path, index=False)

print(f"Saved standardized file: {out_path}")
print(df_standardized.head(3))

In [169]:
# euroengineer jobs
df_euroengineerjobs = pd.read_csv("/Users/HP/Documents/scrapping/filtered_jobs_evp/euroengineer_evp.csv")
df_euroengineerjobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   job_id    148 non-null    int64 
 1   title     148 non-null    object
 2   company   148 non-null    object
 3   location  148 non-null    object
 4   posted    148 non-null    object
 5   summary   148 non-null    object
 6   job_url   148 non-null    object
 7   tags      148 non-null    object
 8   source    148 non-null    object
dtypes: int64(1), object(8)
memory usage: 10.5+ KB


In [ ]:
# canonical conversion for euroengineer_evp.csv from euroengineerjobs website
import pandas as pd
from pathlib import Path

out_path = standardized_folder
# --- 1) Load one source CSV ---
source_path = Path("/Users/HP/Documents/scrapping/filtered_jobs_evp/euroengineer_evp.csv")
df = pd.read_csv(source_path)

# --- 2) Rename/map source columns to canonical names ---
# Example mapping for euroengineerjobs_evp.csv
rename_map = {
    "summary": "description"
}
df = df.rename(columns=rename_map)

# Optional metadata columns
df["source"] = "euroengineerjobs"
df["raw_source_file"] = source_path.name

# canonical schema 
canonical_cols = [
    "job_url",
    "source",
    "title",
    "company",
    "location",
    "description",
    "seniority",
    "tags",
    "raw_source_file"
]

# --- 3) Add missing canonical columns as empty values ---
for col in canonical_cols:
    if col not in df.columns:
        df[col] = pd.NA

# --- 4) Reorder columns to canonical order and save as *_standardized.csv ---
df_standardized = df[canonical_cols]
out_path = standardized_folder / f"{source_path.stem}_std.csv"
df_standardized.to_csv(out_path, index=False)

print(f"Saved standardized file: {out_path}")
print(df_standardized.head(3))

In [ ]:
# euroengineer jobs; energyjobsearch; euroclimatejobs; rejobs [ALL CLEAN AND STANDARDIZED]
df_euroengineerjobs = pd.read_csv("/Users/HP/Documents/scrapping/standardized_jobs/rejobs_evp_std.csv")
df_euroengineerjobs.info()

In [182]:
# merge all standardized files into one dataframe and save to csv
import pandas as pd
from pathlib import Path
import os
#make merged_jobs folder if it doesn't exist
merged_folder = Path("/Users/HP/Documents/scrapping/merged_jobs")
merged_folder.mkdir(exist_ok=True)

standardized_folder = Path("/Users/HP/Documents/scrapping/standardized_jobs")
all_files = list(standardized_folder.glob("*_std.csv"))

df_merged = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
output_csv = merged_folder / "merged_jobs.csv"
df_merged.to_csv(output_csv, index=False)

In [184]:
# view data sctructure of the merged csv
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2773 entries, 0 to 2772
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   job_url          2773 non-null   object
 1   source           2773 non-null   object
 2   title            2773 non-null   object
 3   company          2773 non-null   object
 4   location         2766 non-null   object
 5   description      2773 non-null   object
 6   seniority        1689 non-null   object
 7   tags             2773 non-null   object
 8   raw_source_file  2773 non-null   object
dtypes: object(9)
memory usage: 195.1+ KB


In [186]:
# inspection of duplicate rows in merged csv {size: 2773 rows}
import pandas as pd
from pathlib import Path

# Load merged data
path = Path("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs.csv")
df = pd.read_csv(path)

# Duplicate definition used in audit
dup_keys = ["source", "title", "company", "location"]

# 1) All rows that belong to duplicate groups (includes first occurrence)
dup_rows = df[df.duplicated(subset=dup_keys, keep=False)].copy()

# 2) Sort for easy inspection
dup_rows = dup_rows.sort_values(dup_keys).reset_index(drop=True)

#print(f"Duplicate rows (all members of duplicate groups): {len(dup_rows)}")
#print(f"Duplicate groups: {dup_rows.groupby(dup_keys).ngroups}")

# Show first rows to inspect in notebook
#dup_rows.head(50)
#identify sources of duplicates
dup_sources = dup_rows["source"].value_counts()
print("Duplicate rows by source:")
print(dup_sources)



Duplicate rows by source:
source
energyjobsearch    89
rejobs             36
Name: count, dtype: int64


In [187]:
#save duplicate rows to csv for inspection
# Optional: save duplicate rows for manual review
out_path = Path("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs_duplicate.csv")
dup_rows.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
    

Saved: /Users/HP/Documents/scrapping/merged_jobs/merged_jobs_duplicate.csv


In [ ]:
# remove duplicate rows
deduped = df.drop_duplicates(subset=dup_keys, keep="first", inplace=False)  # or keep="last"

#save deduplicated data to new csv
deduped_path = Path("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs_no_duplicates.csv")
deduped.to_csv(deduped_path, index=False)

deduped.info()

In [ ]:
#verify that duplicates are removed
dup_keys = ["source", "title", "company", "location"]
df2 = pd.read_csv("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs_no_duplicates.csv")
print(df2.duplicated(subset=dup_keys).sum())
print(len(df2))
df2.info()


In [ ]:
#check which sources have no seniority
df_seniority = df2[df2["seniority"].isna()]["source"].value_counts()
print("Sources with missing seniority:")
print(df_seniority)

In [ ]:
# what are common words used in the seniority column where source == "energyjobsearch"
df_energyjobsearch_seniority = df2[df2["source"] == "energyjobsearch"]["seniority"].dropna()
print("Unique seniority values in energyjobsearch:")
print(df_energyjobsearch_seniority.unique())

# words which are not common in seniority column of energyjobsearch
uncommon_seniority = df_energyjobsearch_seniority[~df_energyjobsearch_seniority.isin(["entry level", "mid level", "senior level", "manager", "executive"])]
print("Uncommon seniority values in energyjobsearch:")
print(uncommon_seniority.unique())

#count distinct seniority values in energyjobsearch 
distinct_seniority_count = df_energyjobsearch_seniority.nunique()
print(f"Number of distinct seniority values in energyjobsearch: {distinct_seniority_count}")

Unique seniority values in energyjobsearch:
['MID_LEVEL' 'NOT_APPLICABLE' 'SENIOR']
Uncommon seniority values in energyjobsearch:
['MID_LEVEL' 'NOT_APPLICABLE' 'SENIOR']
Number of distinct seniority values in energyjobsearch: 3


In [214]:
#solving the missing seniority from:euroclimatejobs, euroengineerjobs, rejobs. 
# the energysearchjobs has 100% seniority filled
import re
import pandas as pd

# Compile once for speed and consistency
SENIORITY_RULES = [
    # Executive / leadership
    ("EXECUTIVE", re.compile(r"\b(c[-\s]?level|policy|regulations?|chief|vp|vice president|director|head of|executive|gm|general manager)\b", re.I)),
    # Senior
    ("SENIOR", re.compile(r"\b(senior|sr\.?|lead|policy|regulations?|principal|staff engineer|expert|specialist|specialist ii|specialist iii|manager)\b", re.I)),
    # Mid-level
    ("MID_LEVEL", re.compile(r"\b(mid[-\s]?level|policy|regulations?|intermediate|associate|engineer|engineer ii|specialist|specialist i|experienced)\b", re.I)),
    # Entry / junior
    ("ENTRY_JUNIOR", re.compile(r"\b(junior|jr\.?|entry[-\s]?level|graduate|regulations?|policy|trainee|apprentice|assistant)\b", re.I)),
    # Internship / student
    ("INTERNSHIP", re.compile(r"\b(intern(ship)?|co[-\s]?op|student|undergraduate|regulations?|policy|masters|phd|doctoral|postdoc)\b", re.I)),
]

def infer_seniority(description: str, title: str = "") -> str:
    """
    Returns one of:
    EXECUTIVE, SENIOR, MID_LEVEL, ENTRY_JUNIOR, INTERNSHIP, NOT_SPECIFIED
    """
    text = f"{title or ''} || {description or ''}".strip()
    if not text:
        return "NOT_SPECIFIED"

    # priority order matters (exec > senior > mid > junior > internship)
    for label, pattern in SENIORITY_RULES:
        if pattern.search(text):
            return label

    return "NOT_SPECIFIED"

# df has columns: description, title
df["seniority_inferred"] = df.apply(
    lambda r: infer_seniority(r.get("description", ""), r.get("title", "")),
    axis=1
)

# --- 2. Fill Missing Values ---
# Use the inferred values to fill NaNs in the original 'seniority' column
df["seniority"].fillna(df["seniority_inferred"], inplace=True)

# quick check
print(df["seniority_inferred"].value_counts(dropna=False))



seniority_inferred
SENIOR           1573
NOT_SPECIFIED     542
MID_LEVEL         444
EXECUTIVE         113
INTERNSHIP         67
ENTRY_JUNIOR       34
Name: count, dtype: int64


/var/folders/kr/646f6c153fv3167lv25vd8y40000gn/T/ipykernel_25109/2249079030.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["seniority"].fillna(df["seniority_inferred"], inplace=True)


In [215]:
# --- 3. Save Final Dataset ---
# Save the fully processed data to a new file
final_output_path = Path("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs_seniority.csv")
df.to_csv(final_output_path, index=False)

# --- 4. Verification ---
print(f"Saved final dataset to: {final_output_path}")
print("\nFinal seniority distribution:")
print(df["seniority"].value_counts(dropna=False))

Saved final dataset to: /Users/HP/Documents/scrapping/merged_jobs/merged_jobs_seniority.csv

Final seniority distribution:
seniority
NOT_APPLICABLE    1677
SENIOR             478
NOT_SPECIFIED      239
MID_LEVEL          221
EXECUTIVE           92
INTERNSHIP          45
ENTRY_JUNIOR        21
Name: count, dtype: int64


In [212]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2773 entries, 0 to 2772
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   job_url             2773 non-null   object
 1   source              2773 non-null   object
 2   title               2773 non-null   object
 3   company             2773 non-null   object
 4   location            2766 non-null   object
 5   description         2773 non-null   object
 6   seniority           2773 non-null   object
 7   tags                2773 non-null   object
 8   raw_source_file     2773 non-null   object
 9   seniority_inferred  2773 non-null   object
dtypes: object(10)
memory usage: 216.8+ KB


In [ ]:
# identify rows with seniority "NOT_SPECIFIED"
not_specified_seniority = df[df["seniority"] == "NOT_SPECIFIED"]
print(f"Rows with seniority 'NOT_SPECIFIED': {len(not_specified_seniority)}")

# sources of rows with seniority "NOT_SPECIFIED"
not_specified_sources = not_specified_seniority["source"].value_counts()
print("Sources with seniority 'NOT_SPECIFIED':")
print(not_specified_sources)

#save rows with seniority "NOT_SPECIFIED" to csv for inspection
not_specified_path = Path("/Users/HP/Documents/scrapping/merged_jobs/seniority_not_specified.csv")
not_specified_seniority.to_csv(not_specified_path, index=False)
print(f"Saved rows with seniority 'NOT_SPECIFIED' to: {not_specified_path}")


Rows with seniority 'NOT_SPECIFIED': 239
Sources with seniority 'NOT_SPECIFIED':
source
rejobs             225
euroclimatejobs     14
Name: count, dtype: int64
Saved rows with seniority 'NOT_SPECIFIED' to: /Users/HP/Documents/scrapping/merged_jobs/seniority_not_specified.csv


In [ ]:
# info for rows with seniority "NOT_SPECIFIED" csv
df_not_specified = pd.read_csv("/Users/HP/Documents/scrapping/merged_jobs/seniority_not_specified.csv")
df_not_specified.info()

In [221]:
# remove rows with seniority "NOT_SPECIFIED" and save to new csv
df_final = df[df["seniority"] != "NOT_SPECIFIED"].reset_index(drop=True)
final_no_not_specified_path = Path("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs_final.csv")
df_final.to_csv(final_no_not_specified_path, index=False)
print(f"Saved final dataset without 'NOT_SPECIFIED' rows in seniority to: {final_no_not_specified_path}")

Saved final dataset without 'NOT_SPECIFIED' rows in seniority to: /Users/HP/Documents/scrapping/merged_jobs/merged_jobs_final.csv


In [222]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2534 entries, 0 to 2533
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   job_url             2534 non-null   object
 1   source              2534 non-null   object
 2   title               2534 non-null   object
 3   company             2534 non-null   object
 4   location            2530 non-null   object
 5   description         2534 non-null   object
 6   seniority           2534 non-null   object
 7   tags                2534 non-null   object
 8   raw_source_file     2534 non-null   object
 9   seniority_inferred  2534 non-null   object
dtypes: object(10)
memory usage: 198.1+ KB


In [ ]:
# drop rows with no values in location column and save to new csv
df_final_no_location = df_final.dropna(subset=["location"]).reset_index(drop=True)
final_no_location_path = Path("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs_final_no_location.csv")
df_final_no_location.to_csv(final_no_location_path, index=False)
print(f"Saved final dataset without missing location to: {final_no_location_path}")

# also save rows with missing location to a separate csv for inspection
df_final_missing_location = df_final[df_final["location"].isna()].reset_index(drop=True)
missing_location_path = Path("/Users/HP/Documents/scrapping/merged_jobs/merged_jobs_final_missing_location.csv")
df_final_missing_location.to_csv(missing_location_path, index=False)
print(f"Saved rows with missing location to: {missing_location_path}")

# info for the two datasets
print("Final dataset without missing location:")
df_final_no_location.info()
print("\nFinal dataset with missing location:")
df_final_missing_location.info()


In [224]:
# add a new column name it "expert_role"
import re

# 1) Fill these lists with your final terms
sme_terms = [
    "power systems engineer", "energy systems engineer", 
    "grid engineer", "electrical engineer energy", 
    "HVDC engineer", "smart grid engineer", "renewable energy engineer", 
    "energy storage engineer", "battery engineer", 
    "control systems engineer energy", "load flow analysis", 
    "grid integration", "power electronics", "SCADA energy", 
    "distribution systems operator", "transmission planning", "microgrid engineer"
]

generalist_terms = [
    "energy analyst", "energy consultant", "energy systems analyst",
    "energy project manager", "energy program manager", "sustainability consultant",
    "energy advisor", "techno-economic analyst energy", "energy market analyst",
    "energy strategy consultant", "energy transition analyst", "systems integration energy",
    "energy transition", "decarbonization", "integrated energy systems"

]

normative_terms = [
    "energy policy analyst", "energy policy advisor", "climate policy expert",
    "energy regulator", "policy officer energy", "public affairs energy", 
    "regulatory affairs energy", "energy governance", "policy consultant energy",
    "EU energy policy", "climate governance", "just transition", "energy justice", 
    "sustainability policy", "net zero strategy", "climate strategy"
]

def build_pattern(terms):
    if not terms:
        return None
    escaped = [re.escape(t.strip()) for t in terms if t and t.strip()]
    if not escaped:
        return None
    # match any term as a phrase, case-insensitive
    return re.compile(r"\b(" + "|".join(escaped) + r")\b", re.IGNORECASE)

SME_PAT = build_pattern(sme_terms)
GENERALIST_PAT = build_pattern(generalist_terms)
NORMATIVE_PAT = build_pattern(normative_terms)

def assign_expert_role(title: str) -> str:
    t = str(title or "").strip()
    if not t:
        return "no_defined_roles"

    # Priority order (change if you want different precedence)
    if SME_PAT and SME_PAT.search(t):
        return "sme"
    if GENERALIST_PAT and GENERALIST_PAT.search(t):
        return "generalist"
    if NORMATIVE_PAT and NORMATIVE_PAT.search(t):
        return "normative"
    return "no_defined_roles"

# 2) Apply to dataframe
df["expert_role"] = df["title"].apply(assign_expert_role)

# 3) Quick check
print(df["expert_role"].value_counts(dropna=False))


expert_role
no_defined_roles    2746
generalist            18
sme                    9
Name: count, dtype: int64


In [ ]:
# Improved expert_role assignment with:
# - broader keyword coverage
# - title + description + tags matching
# - weighted scoring
# - precedence (normative > sme > generalist)
# - reason tracking

import re
import pandas as pd
import os

#create final directory if it doesn't exist
final_dir = Path("/Users/HP/Documents/scrapping/working_dir")
final_dir.mkdir(exist_ok=True)

# -----------------------------
# 1) Keyword dictionaries
# -----------------------------
SME_TERMS = [
    "engineer", "engineering", "electrical", "mechanical", "hvdc", "scada",
    "power systems", "grid", "microgrid", "transmission", "distribution",
    "battery", "energy storage", "power electronics", "controls", "automation",
    "protection", "load flow", "integration", "operations"
]

GENERALIST_TERMS = [
    "analyst", "consultant", "advisor", "specialist", "coordinator",
    "project manager", "program manager", "product manager", "strategy",
    "market", "business development", "commercial", "sustainability",
    "transition", "decarbonization", "decarbonisation"
]

NORMATIVE_TERMS = [
    "policy", "regulatory", "regulation", "compliance", "governance",
    "public affairs", "advocacy", "standards", "framework", "justice",
    "just transition", "net zero", "climate strategy", "energy law",
    "permitting", "esg", "social", "community engagement"
]

# -----------------------------
# 2) Compile regex patterns
# -----------------------------
def compile_terms(terms):
    terms = [t.strip().lower() for t in terms if t and t.strip()]
    # phrase-safe contains matching with word boundaries
    return [re.compile(rf"\b{re.escape(t)}\b", re.IGNORECASE) for t in terms]

PATTTERNS = {
    "sme": compile_terms(SME_TERMS),
    "generalist": compile_terms(GENERALIST_TERMS),
    "normative": compile_terms(NORMATIVE_TERMS),
}

# -----------------------------
# 3) Helpers
# -----------------------------
def norm_text(x):
    s = "" if pd.isna(x) else str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def score_role(text, patterns):
    hits = []
    score = 0
    for pat in patterns:
        m = pat.search(text)
        if m:
            hits.append(m.group(0).lower())
            score += 1
    return score, sorted(set(hits))

def assign_role(row, min_score=1):
    title = norm_text(row.get("title"))
    desc = norm_text(row.get("description"))
    tags = norm_text(row.get("tags"))

    # weighted text: title higher signal than desc/tags
    # (implemented by scoring separately)
    scores = {}
    reasons = {}
    for role in ["normative", "sme", "generalist"]:
        s_title, h_title = score_role(title, PATTTERNS[role])
        s_desc, h_desc = score_role(desc, PATTTERNS[role])
        s_tags, h_tags = score_role(tags, PATTTERNS[role])

        weighted = (2 * s_title) + (1 * s_desc) + (1 * s_tags)
        hits = sorted(set(h_title + h_desc + h_tags))
        scores[role] = weighted
        reasons[role] = hits

    # precedence in tie: normative > sme > generalist
    ordered_roles = ["normative", "sme", "generalist"]
    best_role = max(ordered_roles, key=lambda r: scores[r])
    best_score = scores[best_role]

    if best_score < min_score:
        return pd.Series(["no_defined_roles", 0, ""])

    return pd.Series([best_role, best_score, ", ".join(reasons[best_role][:8])])

# -----------------------------
# 4) Apply to dataframe
# -----------------------------
# expects columns: title, description, tags (if tags missing, code still works)
for col in ["title", "description", "tags"]:
    if col not in df.columns:
        df[col] = ""

df[["expert_role", "expert_role_score", "expert_role_reason"]] = df.apply(assign_role, axis=1)

# -----------------------------
# 5) Quick checks
# -----------------------------
print(df["expert_role"].value_counts(dropna=False))
print(df.groupby("expert_role")["expert_role_score"].describe())
df[["title", "expert_role", "expert_role_score", "expert_role_reason"]].head(20)

# Save final dataset with expert_role
final_output_path = Path("/Users/HP/Documents/scrapping/working_dir/merged_jobs_rolesFinale.csv")
df.to_csv(final_output_path, index=False)
print(f"Saved final dataset with expert_role to: {final_output_path}")

#filter dataset for only rows with expert_role other than "not_defined" and save to new csv
df_filtered = df[df["expert_role"] != "no_defined_roles"]
df_filtered.to_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_filtered.csv", index=False)
print(f"Saved filtered dataset to: {final_output_path}")

expert_role
sme                 1303
no_defined_roles     697
generalist           631
normative            142
Name: count, dtype: int64
                   count      mean       std  min  25%  50%  75%   max
expert_role                                                           
generalist         631.0  2.599049  1.098021  1.0  2.0  2.0  3.0   7.0
no_defined_roles   697.0  0.000000  0.000000  0.0  0.0  0.0  0.0   0.0
normative          142.0  2.190141  1.867729  1.0  1.0  2.0  2.0  10.0
sme               1303.0  3.947045  2.360606  1.0  2.0  3.0  5.0  13.0
Saved final dataset with expert_role to: /Users/HP/Documents/scrapping/working_dir/merged_jobs_rolesFinale.csv
Saved filtered dataset to: /Users/HP/Documents/scrapping/working_dir/merged_jobs_rolesFinale.csv


In [291]:
df = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_filtered.csv")
df.info()   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2076 entries, 0 to 2075
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   job_url             2076 non-null   object
 1   source              2076 non-null   object
 2   title               2076 non-null   object
 3   company             2076 non-null   object
 4   location            2069 non-null   object
 5   description         2076 non-null   object
 6   seniority           2076 non-null   object
 7   tags                2076 non-null   object
 8   raw_source_file     2076 non-null   object
 9   seniority_inferred  2076 non-null   object
 10  expert_role         2076 non-null   object
 11  expert_role_score   2076 non-null   int64 
 12  expert_role_reason  2076 non-null   object
dtypes: int64(1), object(12)
memory usage: 211.0+ KB


In [292]:
# select rows with expert_role "sme" and save to new csv
df_sme = df[df["expert_role"] == "sme"]
df_sme.to_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_sme.csv", index=False)
df_sme.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1303 entries, 7 to 2074
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   job_url             1303 non-null   object
 1   source              1303 non-null   object
 2   title               1303 non-null   object
 3   company             1303 non-null   object
 4   location            1300 non-null   object
 5   description         1303 non-null   object
 6   seniority           1303 non-null   object
 7   tags                1303 non-null   object
 8   raw_source_file     1303 non-null   object
 9   seniority_inferred  1303 non-null   object
 10  expert_role         1303 non-null   object
 11  expert_role_score   1303 non-null   int64 
 12  expert_role_reason  1303 non-null   object
dtypes: int64(1), object(12)
memory usage: 142.5+ KB


In [293]:
# select rows with expert_role "generalist" and save to new csv
df_generalist = df[df["expert_role"] == "generalist"]
df_generalist.to_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_generalist.csv", index=False)
df_generalist.info()

<class 'pandas.core.frame.DataFrame'>
Index: 631 entries, 0 to 2075
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   job_url             631 non-null    object
 1   source              631 non-null    object
 2   title               631 non-null    object
 3   company             631 non-null    object
 4   location            627 non-null    object
 5   description         631 non-null    object
 6   seniority           631 non-null    object
 7   tags                631 non-null    object
 8   raw_source_file     631 non-null    object
 9   seniority_inferred  631 non-null    object
 10  expert_role         631 non-null    object
 11  expert_role_score   631 non-null    int64 
 12  expert_role_reason  631 non-null    object
dtypes: int64(1), object(12)
memory usage: 69.0+ KB


In [ ]:
# select rows with expert_role "normative" and save to new csv
df_normative = df[df["expert_role"] == "normative"]
df_normative.to_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative.csv", index=False)
df_normative.head(50)

In [295]:
# count rows group by them having similar values in values of title column
title_counts_norm = (df_normative["title"].fillna("").str.strip().str.lower().value_counts())
#print("Top 30 most common job titles:")
#print(title_counts_norm.head(30))

# how many distinct job titles are there in normative roles?
distinct_titles_norm = df_normative["title"].nunique()
print(f"Number of distinct job titles in normative roles: {distinct_titles_norm}")

# how many titles contains word "policy" or "consultant" or "regulation" in normative roles?
policy_titles_norm = df_normative[df_normative["title"].str.contains("policy|consultant|regulation", case=False, na=False)]
print(f"Number of job titles containing 'policy', 'consultant', or 'regulation' in normative roles: {len(policy_titles_norm)}")


Number of distinct job titles in normative roles: 134
Number of job titles containing 'policy', 'consultant', or 'regulation' in normative roles: 5


In [296]:
# drop the rows which its title contains the following words.
df = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative.csv")
drop_keywords = ["engineer", "2026", "intern"]
pattern = re.compile(r"\b(" + "|".join(drop_keywords) + r")\b", re.IGNORECASE)
df_no_interns = df[~df["title"].str.contains(pattern, na=False)].reset_index(drop=True)
print(f"Rows after dropping intern/assistant positions: {len(df_no_interns)}")

# save to new csv
final_no_interns_path = Path("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns.csv")
df_no_interns.to_csv(final_no_interns_path, index=False)
print(f"Saved final dataset without intern/assistant positions to: {final_no_interns_path}")
df_no_interns.info()

Rows after dropping intern/assistant positions: 127
Saved final dataset without intern/assistant positions to: /Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   job_url             127 non-null    object
 1   source              127 non-null    object
 2   title               127 non-null    object
 3   company             127 non-null    object
 4   location            127 non-null    object
 5   description         127 non-null    object
 6   seniority           127 non-null    object
 7   tags                127 non-null    object
 8   raw_source_file     127 non-null    object
 9   seniority_inferred  127 non-null    object
 10  expert_role         127 non-null    object
 11  expert_role_score   127 non-null    int64 
 12  expert_role_reason  127

/var/folders/kr/646f6c153fv3167lv25vd8y40000gn/T/ipykernel_25109/3132465434.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_no_interns = df[~df["title"].str.contains(pattern, na=False)].reset_index(drop=True)


In [297]:
# count rows in: "/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns.csv", that have more than 25 words in the 'description' column
df_no_interns = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns.csv")
df_no_interns["description_word_count"] = df_no_interns["description"].fillna("").apply(lambda x: len(str(x).split()))
long_descriptions = df_no_interns[df_no_interns["description_word_count"] > 25]

# print count of rows with more than 25 words in description
print(f"Number of rows with more than 25 words in description: {len(long_descriptions)}")

long_descriptions.head(25)

# save rows with more than 25 words in description to new csv
long_descriptions_path = Path("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns_long_descriptions.csv")
long_descriptions.to_csv(long_descriptions_path, index=False)
print(f"Saved rows with more than 25 words in description to: {long_descriptions_path}")

Number of rows with more than 25 words in description: 23
Saved rows with more than 25 words in description to: /Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns_long_descriptions.csv


In [ ]:
long_descriptions = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns_long_descriptions.csv")
long_descriptions.head(20)

In [286]:
# count how many rows are senior level in the normative roles dataset without interns
long_descriptions = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns_long_descriptions.csv")
senior_count = (long_descriptions['seniority'].astype(str).str.strip().str.upper().isin(['EXECUTIVE', 'SENIOR']).sum())

# print count of senior level rows
print(f"Number of senior level rows in normative roles with long descriptions: {senior_count}")

Number of senior level rows in normative roles with long descriptions: 8


In [289]:
# count how many rows are mid-level in the normative roles dataset without interns
long_descriptions = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_normative_no_interns_long_descriptions.csv")
mid_level_count = (long_descriptions['seniority'].astype(str).str.strip().str.upper().isin(['MID_LEVEL']).sum())

# print count of mid-level rows
print(f"Number of mid-level rows in normative roles with long descriptions: {mid_level_count}")

Number of mid-level rows in normative roles with long descriptions: 1


In [298]:
import pandas as pd
from pathlib import Path

in_path = Path("/Users/HP/Documents/scrapping/working_dir/merged_generalist_clean_no_flagged.csv")
df = pd.read_csv(in_path)

# True if job_url contains "other"
mask_other = df["job_url"].fillna("").str.contains("other", case=False, na=False)

df_no_other = df[~mask_other].copy()
df_has_other = df[mask_other].copy()

out_no_other = in_path.with_name("merged_generalist_clean_no_flagged_url_no_other.csv")
out_has_other = in_path.with_name("merged_generalist_clean_no_flagged_url_has_other.csv")

df_no_other.to_csv(out_no_other, index=False)
df_has_other.to_csv(out_has_other, index=False)

print(f"Input rows: {len(df)}")
print(f"No 'other': {len(df_no_other)} -> {out_no_other}")
print(f"Has 'other': {len(df_has_other)} -> {out_has_other}")
print(f"Check total: {len(df_no_other) + len(df_has_other)}")

Input rows: 395
No 'other': 243 -> /Users/HP/Documents/scrapping/working_dir/merged_generalist_clean_no_flagged_url_no_other.csv
Has 'other': 152 -> /Users/HP/Documents/scrapping/working_dir/merged_generalist_clean_no_flagged_url_has_other.csv
Check total: 395


In [300]:
import pandas as pd
from pathlib import Path

df_no_other_senior = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_generalist_clean_no_flagged_url_no_other.csv")

# filter rows which their title columns contains "senior" or "executive" or "manager"
mask_senior = df_no_other_senior["title"].fillna("").str.contains("senior|executive|manager", case=False, na=False)
df_no_other_senior_filtered = df_no_other_senior[mask_senior].copy()
#print count of rows after filtering
print(f"Rows with 'senior', 'executive', or 'manager' in title: {len(df_no_other_senior_filtered)}")

# for all rows which their title columns contains "senior" or "executive" or "manager", change the value in seniority column to "SENIOR" else "MID_LEVEL"
df_no_other_senior["seniority"] = df_no_other_senior["title"].apply(lambda x: "SENIOR" if pd.notna(x) and re.search(r"\b(senior|executive|manager)\b", str(x), re.IGNORECASE) else "MID_LEVEL")
# save to the same csv file
df_no_other_senior.to_csv("/Users/HP/Documents/scrapping/working_dir/merged_generalist_clean_no_flagged_url_no_other.csv", index=False)
print(f"Updated seniority based on title and saved to: /Users/HP/Documents/scrapping/working_dir/merged_generalist_clean_no_flagged_url_no_other.csv")


Rows with 'senior', 'executive', or 'manager' in title: 61
Updated seniority based on title and saved to: /Users/HP/Documents/scrapping/working_dir/merged_generalist_clean_no_flagged_url_no_other.csv


In [ ]:
# does the assigned roles really mean the specified role?
import pandas as pd
import os 
from sklearn.metrics import confusion_matrix, classification_report

#read csv
df = pd.read_csv("/Users/HP/Documents/scrapping/working_dir/merged_jobsRoles_filtered.csv")

# 1) Start from your full dataframe (must contain expert_role)

# Keep only rows with assigned roles you want to validate
roles = ["sme", "generalist", "normative"]
val = df[df["expert_role"].isin(roles)].copy()

# 2) Draw balanced sample for manual labeling (e.g., 80 each)
n_per_role = 100
sample = (
    val.groupby("expert_role", group_keys=False)
       .apply(lambda x: x.sample(min(len(x), n_per_role), random_state=42))
       .reset_index(drop=True)
)

# Add empty manual label column for you to fill
sample["manual_role"] = ""

# Save for manual annotation
sample_path = "/Users/HP/Documents/scrapping/validation_sample_roles.csv"
sample.to_csv(sample_path, index=False)
print("Label this file manually, fill `manual_role`, then reload it:", sample_path)


In [302]:
## cleaning the sme data
## remove those rows which their url has 'other' in it:
import pandas as pd
from pathlib import Path

in_path = Path("/Users/HP/Documents/scrapping/working_dir/merged_jobsroles_sme.csv")
df = pd.read_csv(in_path)

# True if job_url contains "other"
mask_other = df["job_url"].fillna("").str.contains("other", case=False, na=False)

df_no_other = df[~mask_other].copy()
df_has_other = df[mask_other].copy()

out_no_other = in_path.with_name("merged_jobsroles_sme_no_other.csv")
out_has_other = in_path.with_name("merged_jobroles_sme_has_other.csv")

df_no_other.to_csv(out_no_other, index=False)
df_has_other.to_csv(out_has_other, index=False)

print(f"Input rows: {len(df)}")
print(f"No 'other': {len(df_no_other)} -> {out_no_other}")
print(f"Has 'other': {len(df_has_other)} -> {out_has_other}")
print(f"Check total: {len(df_no_other) + len(df_has_other)}")

Input rows: 1303
No 'other': 1009 -> /Users/HP/Documents/scrapping/working_dir/merged_jobsroles_sme_no_other.csv
Has 'other': 294 -> /Users/HP/Documents/scrapping/working_dir/merged_jobroles_sme_has_other.csv
Check total: 1303
